# Deploying NVIDIA Nemotron-3-Ultra with SGLang

This notebook will walk you through how to run the `nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B` model with SGLang.

[SGLang](https://github.com/sgl-project/sglang) is a fast serving framework for large language models and vision language models.

For more details on the model [click here](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16)

Prerequisites:
- NVIDIA GPUs with recent drivers and CUDA 12.x
  - BF16: 8x GB200/B200/B300, 16x H100, 8x H200 | NVFP4: 4x GB200/B200/B300/GB300, 8x H100
- Python 3.10+
- Docker


## Overview

- **Serve** the Nemotron-3-Ultra model using SGLang
- **Query the model** through an OpenAI-compatible REST API
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget

## Table of Contents

1. **Environment setup** - Dependencies and virtual environment
2. **Verify GPU** - Confirm CUDA and GPU availability
3. **Start SGLang Server** - Launch BF16 or NVFP4 variant
4. **Generate responses** - Chat completions and streaming
   - **Simple vs streamed generation** - Basic and streamed requests
   - **Reasoning** - Thinking mode examples
   - **Tool calling** - Function calling via OpenAI tools schema
   - **Controlling Reasoning Budget** - Limit reasoning trace length
5. **Cleanup and shutdown**

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). For the easiest way to get started, check out the NVFP4 variant on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

**For NVFP4 (8xH100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3EPQszEEuDqceuEswhpqhZRrd1M)

## Environment setup

### Install client dependencies

The SGLang server runs inside a Docker container. This notebook only needs the OpenAI client and tokenizer utilities installed.

If you are not on Brev, it is recommended to create and activate a virtual environment first:

```shell
python3 -m venv .venv
source .venv/bin/activate
```

> **Brev users:** A virtual environment is already set up — skip the step above and run the cells below directly.

In [ ]:
#If pip not found
!python3 -m ensurepip --default-pip

In [ ]:
%pip install openai transformers torch

## Verify GPU

Confirm CUDA is available and your GPUs are visible to PyTorch.

> **Expected output:** `CUDA available: True` with 8 GPUs listed (B200 for BF16, H100 for NVFP4). If CUDA is `False`, check your driver installation.

In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

CUDA available: True
Num GPUs: 8
GPU[0]: NVIDIA B200
GPU[1]: NVIDIA B200
GPU[2]: NVIDIA B200
GPU[3]: NVIDIA B200
GPU[4]: NVIDIA B200
GPU[5]: NVIDIA B200
GPU[6]: NVIDIA B200
GPU[7]: NVIDIA B200


## Start SGLang Server

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the SGLang container. The `--network=host` flag makes the server reachable at `localhost:8000` from the notebook.

```shell
docker run --rm -it \
  --gpus all \
  --cap-add SYS_NICE \
  --ipc=host \
  --network=host \
  --shm-size=16g \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  -e SAFETENSORS_FAST_GPU=1 \
  -e NVIDIA_TF32_OVERRIDE=1 \
  -e SGLANG_DISABLE_DEEP_GEMM=1 \
  --entrypoint /bin/bash \
  lmsysorg/sglang:v0.5.12.post1
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

All `python3 -m sglang.launch_server` commands below should be run from inside this container.

### Configuration reference

| | BF16 | NVFP4 |
|---|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16` | `nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-NVFP4` |
| **Supported hardware** | 8x GB200/B200/B300, 16x H100, 8x H200 | 4x GB200/B200/B300/GB300, 8x H100 |
| **Docker image** | `lmsysorg/sglang:v0.5.12.post1` | `lmsysorg/sglang:v0.5.12.post1` |
| **TP / EP (this notebook)** | 8 / 8 | 8 / — |
| **Reasoning parser** | `nemotron_3` | `nemotron_3` |
| **Tool parser** | `qwen3_coder` | `qwen3_coder` |
| **Port** | 8000 | 8000 |

### Start server

Run one of the following commands from inside the Docker container terminal.

> **Note:** Parser names are backend-specific and not interchangeable. SGLang uses `--reasoning-parser nemotron_3` and `--tool-call-parser qwen3_coder`. vLLM uses `--reasoning-parser nemotron_v3` and TensorRT-LLM uses `--reasoning_parser nano-v3` — these are different identifiers for the same logical capability.

#### BF16 (8x B200)

```shell
python3 -m sglang.launch_server \
  --model-path nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16 \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name nemotron-3-ultra \
  --trust-remote-code \
  --tensor-parallel-size 8 \
  --expert-parallel-size 8 \
  --disable-piecewise-cuda-graph \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

#### NVFP4 (8x H100)

> **Note:** Expert parallelism is not used on H100 — TP8+EP OOMs on this architecture, so TP8 without EP is the correct configuration.

```shell
python3 -m sglang.launch_server \
  --model-path nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-NVFP4 \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name nemotron-3-ultra \
  --trust-remote-code \
  --tensor-parallel-size 8 \
  --reasoning-parser nemotron_3 \
  --tool-call-parser qwen3_coder
```

#### Optional performance flags

**MTP (Multi-Token Prediction):** Speculative decoding via EAGLE predicts multiple tokens per step, improving throughput. Add the following flags to either serve command, and set `SGLANG_ENABLE_SPEC_V2=1` in the environment before running — both must be used together:

```
--speculative-algorithm EAGLE
--speculative-num-steps 5
--speculative-eagle-topk 1
--speculative-num-draft-tokens 5
--mamba-scheduler-strategy extra_buffer
```

**OOM workaround:** If you run into an out-of-memory error during startup (typically during CUDA graph capture), add the following flag to the serve command:

```
--disable-piecewise-cuda-graph
```

### Wait for the server to be ready

After starting the container, poll `/v1/models` before sending any requests.

Run the following in a host terminal:

```shell
until curl -sf http://localhost:8000/v1/models > /dev/null 2>&1; do
  echo "Waiting for server..."; sleep 10
done
echo "Server is ready"
```

> **Expected output:** The loop prints `Waiting for server...` while shards load, then exits and prints `Server is ready` once the server is accepting requests.

> **Note:** On the first run, the model weights will be downloaded from Hugging Face before loading begins. For a large model like this, the combined download and load time can be significant — this is expected behavior.

## Generate responses

### Client setup

In [1]:
from openai import OpenAI

# Set this to the model you launched the server with
SERVED_MODEL_NAME = "nemotron-3-ultra" 
BASE_URL = "http://localhost:8000/v1"
API_KEY = "EMPTY"  # SGLang does not require an API key by default

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(f"OpenAI client configured to use server at: {BASE_URL}")
print(f"Using model: {SERVED_MODEL_NAME}")

OpenAI client configured to use server at: http://localhost:8000/v1
Using model: nemotron-3-ultra


### Simple vs streamed generation

In [2]:
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "Give me 3 bullet points about SGLang."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
)
print("Reasoning:", resp.choices[0].message.reasoning_content)
print("Content:", resp.choices[0].message.content)

Reasoning: The user wants 3 bullet points about SGLang.
Content: *   **RadixAttention for Prefix Caching:** SGLang features a unique Radix Tree-based attention backend (RadixAttention) that automatically shares KV cache across requests with common prefixes (e.g., shared system prompts, few-shot examples, or document chunks), significantly reducing memory usage and boosting throughput for multi-turn conversations and RAG workloads.
*   **Structured Generation via Co-Design:** It tightly co-designs the frontend language (a Python-embedded DSL with primitives like `gen`, `select`, `fork`, and `regex`) with the backend runtime. This allows the scheduler to "look ahead" at constrained decoding requirements (like JSON schemas or regex patterns) to optimize token allocation and skip unnecessary logit computations.
*   **High-Performance Distributed Serving:** Built on a disaggregated architecture, SGLang supports tensor parallelism, pipeline parallelism, and data parallelism (via the `DataPar

### Streaming chat completion

In [4]:
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)

The first 5 prime numbers are:

**2, 3, 5, 7, 11**

### Reasoning

> **Note:** The model supports two modes — Reasoning ON (default) and Reasoning OFF. Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`, as shown below.

In [17]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
)
print("Reasoning:", resp.choices[0].message.reasoning_content)
print("Content:", resp.choices[0].message.content)
print()

# Reasoning off
print("Reasoning off")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about SGLang."}
    ],
    temperature=0.2,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False, "force_nonempty_content": True}}
)
print(resp.choices[0].message.content)

Reasoning on
Reasoning: The user wants a haiku about GPUs.
A haiku has a 5-7-5 syllable structure.
Topic: Graphics Processing Units (parallel processing, gaming, AI, rendering).
Content: Silicon arrays,
Parallel dreams render fast,
Pixels bloom to life.

Reasoning off
Here are 3 interesting facts about **SGLang** (Structured Generation Language):

### 1. It uses "RadixAttention" to share KV Cache across requests
This is SGLang’s "secret sauce." In traditional LLM serving, if two users send prompts that share a common prefix (e.g., the same system prompt or few-shot examples), the system recomputes the Key-Value (KV) cache for that prefix separately for every single request.

SGLang implements **RadixAttention**, which organizes the KV cache as a **Radix Tree (Prefix Tree)**. This allows different requests to **share the memory and computation of common prefixes**. In benchmarks involving shared system prompts (like coding agents or RAG), this can increase throughput by **2x to 5x** com

### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`.

> **Note:** When using tool calling with reasoning enabled, you must pass `"force_nonempty_content": true` inside `chat_template_kwargs`. Without it, the server may not correctly parse both the reasoning trace and the tool call output together.

In [19]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    stream=False,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": True,
            "force_nonempty_content": True
        }
    }
)

print("Reasoning:", completion.choices[0].message.reasoning_content)
print("Tool calls:", completion.choices[0].message.tool_calls)

Reasoning: The user asks: "My bill is $50. What will be the amount for 15% tip?" We have a function calculate_tip that takes bill_total and tip_percentage. The bill total is $50. The tip percentage is 15%. So we can compute tip amount using the function. We need to call calculate_tip with bill_total=50 and tip_percentage=15. The function returns presumably the tip amount (maybe total). The description says "Calculate tip" with inputs bill_total and tip_percentage. It doesn't specify what it returns but presumably returns tip amount. We can call it.

Thus, we will call calculate_tip.
Tool calls: [ChatCompletionMessageFunctionToolCall(id='call_0880562c335045f5a9091002', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function', index=0)]


### Controlling Reasoning Budget

The `reasoning_budget` parameter lets you limit how long the model reasons before producing a response. When the reasoning trace reaches the token budget, the model will try to wrap up at the next newline.

> **Note:** If no newline is encountered within 500 tokens after the budget threshold, the reasoning trace is forcibly terminated at `reasoning_budget + 500` tokens.

In [5]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        reasoning_content = response.choices[0].message.reasoning_content or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [4]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B-BF16"  # use actual HF model ID for tokenizer
)

In [6]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    max_tokens=512,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])

Reasoning: The user wants a haiku about GPUs.
A haiku has a 5-7-5 syllable structure..
Content: Massive cores align,  
Crunching math in parallel,  
Pixels bloom to life.


## Cleanup and shutdown

To free resources after this notebook:

1. Stop the SGLang server in the terminal where it was started (`Ctrl+C`).
2. Optionally run the next cell to clear notebook-side CUDA cache.
3. Restart the kernel if needed to ensure a clean state.

In [ ]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Notebook-side CUDA cache cleanup complete.")
print("Primary teardown step: stop the SGLang server with Ctrl+C in its terminal.")